# 06 · Tổng hợp kết quả + phân tích lỗi — Hoàng Duyên (T20/T27/T39) & Đức Tâm (T31/T38)

**Khi nào chạy?** Sau khi **cả 3 bạn đã chạy xong notebook huấn luyện của mình (03, 04, 05) và đã đưa kết quả lên GitHub**
(Phúc đưa `crf.json`, `zeroshot.json`, `du_doan_test.json`). Bạn **không** chạy notebook của người khác — chỉ cần lấy kết quả của họ về.
Chạy sớm hơn cũng không sao: ô 1 sẽ liệt kê còn thiếu file gì, của ai; các ô sau tự bỏ qua phần chưa đủ đầu vào.

| Chạy ở đâu | Cách lấy kết quả của các bạn |
|---|---|
| **VS Code (khuyên dùng)** — mở notebook **trong thư mục repo** `NLP-phoner-covid19-26/notebooks/` | GitHub Desktop → *Fetch origin* → *Pull origin*; dữ liệu đặt ở `data/word`, `data/syllable` |
| **Colab** | Kéo các file `ket_qua_phobert.zip`, `ket_qua_xlmr.zip`, `ket_qua_mbert.zip` (mỗi bạn tải ở ô 10 notebook của mình) và các file của Phúc vào khung **Files** bên trái, rồi chạy ô 0 |

## 0. Chuẩn bị thư mục làm việc

In [ ]:
# @title Bấm ▶ (mở ra nếu muốn xem code)
MA_NGUON = {'configs/common.json': '{\n  "_ghi_chu": "Hợp đồng chung của nhóm. Cả 3 kiến trúc đọc chung file này để so sánh công bằng. Muốn đổi giá trị nào: mở PR + ghi lý do vào docs/QUY_UOC_CHUNG.md, không sửa riêng trong notebook.",\n  "trang_thai": "ĐỀ XUẤT — chốt ở buổi họp 12/10/2026",\n\n  "du_lieu": {\n    "thu_muc": "data",\n    "muc_tu": "data/word",\n    "muc_am_tiet": "data/syllable",\n    "ten_file": "{split}_{muc}.json",\n    "dinh_dang": "Mỗi dòng 1 câu: {\\"words\\": [...], \\"tags\\": [...]}",\n    "so_cau_mong_doi": {"train": 5027, "dev": 2000, "test": 3000}\n  },\n\n  "cac_loai_thuc_the": ["PATIENT_ID", "NAME", "AGE", "GENDER", "JOB", "LOCATION",\n                         "ORGANIZATION", "SYMPTOM_AND_DISEASE", "TRANSPORTATION", "DATE"],\n  "nhan": ["O",\n           "B-PATIENT_ID", "I-PATIENT_ID", "B-NAME", "I-NAME", "B-AGE", "I-AGE",\n           "B-GENDER", "I-GENDER", "B-JOB", "I-JOB", "B-LOCATION", "I-LOCATION",\n           "B-ORGANIZATION", "I-ORGANIZATION", "B-SYMPTOM_AND_DISEASE", "I-SYMPTOM_AND_DISEASE",\n           "B-TRANSPORTATION", "I-TRANSPORTATION", "B-DATE", "I-DATE"],\n\n  "cham_diem": {\n    "thu_vien": "seqeval",\n    "mode": "strict",\n    "scheme": "IOB2",\n    "chi_so_chinh": "f1_micro",\n    "do_lech_chuan": "std mẫu (chia n-1), numpy.std(..., ddof=1)",\n    "chon_checkpoint": "f1_micro cao nhất trên dev; test chỉ dùng để báo cáo"\n  },\n\n  "seeds": [42, 123, 2024],\n\n  "sieu_tham_so": {\n    "learning_rate": 5e-5,\n    "batch_size": 16,\n    "gradient_accumulation_steps": 2,\n    "so_epoch_toi_da": 15,\n    "dung_som_sau": 3,\n    "max_length": 256,\n    "weight_decay": 0.01,\n    "warmup_ratio": 0.1,\n    "fp16": true,\n    "_ly_do": "Bài báo dùng lr 5e-5, batch 32, 30 epoch, dừng sớm 5. Nhóm giữ lr, batch hiệu dụng 16 x 2 = 32, nhưng giảm còn 15 epoch / dừng sớm 3 để kịp chạy 3 seed trước 18/10 trên Colab."\n  },\n\n  "mo_hinh": {\n    "phobert":  {"ten": "vinai/phobert-base",            "muc_du_lieu": "muc_tu",      "nguoi": "Hoàng Duyên"},\n    "xlmr":     {"ten": "xlm-roberta-base",              "muc_du_lieu": "muc_am_tiet", "nguoi": "Đức Tâm"},\n    "mbert":    {"ten": "bert-base-multilingual-cased",  "muc_du_lieu": "muc_am_tiet", "nguoi": "Yến Trinh"},\n    "phobert_khong_tach_tu": {"ten": "vinai/phobert-base", "muc_du_lieu": "muc_am_tiet", "nguoi": "Phúc", "ghi_chu": "Yêu cầu nâng cao (sau BC1)"}\n  },\n\n  "ten_file_ket_qua": "results/{khoa_mo_hinh}_seed{seed}.json  (vd: results/phobert_seed42.json, results/phobert_khong_tach_tu_seed42.json)"\n}\n', 'src/data.py': '"""data.py - Đọc dữ liệu PhoNER_COVID19 và tạo bảng nhãn dùng chung cho cả nhóm.\n\nNgười phụ trách: Đức Tâm · Issue T07\nCách tự kiểm tra (ở thư mục gốc repo):  python src/data.py\nXong khi in ra: train 5027 / dev 2000 / test 3000 câu cho CẢ 2 bản, 21 nhãn, 0 câu lệch số từ - số nhãn.\n\nTiền xử lý duy nhất: chuẩn hóa Unicode NFC cho từng token (nhãn giữ nguyên vì 1 token -> 1 token).\nLý do: mỗi phiên bản dữ liệu gốc có 1.052 token chưa ở dạng NFC (dấu viết tách rời chữ cái) lẫn với dạng dựng sẵn;\ncùng một chữ "Hà" có thể là 2 chuỗi byte khác nhau -> tokenizer cắt khác nhau. NFC là chuẩn an toàn mặc định\ncho văn bản tiếng Việt (bài giảng Session 2 - Words, Tokens; Unicode Standard Annex #15).\n"""\n\nimport json\nimport os\nimport unicodedata\nfrom collections import Counter\n\nTHU_MUC_GOC = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nDUONG_DAN_CAU_HINH = os.path.join(THU_MUC_GOC, "configs", "common.json")\nCAC_TAP = ["train", "dev", "test"]\n\n\ndef doc_cau_hinh(duong_dan=DUONG_DAN_CAU_HINH):\n    """Đọc configs/common.json -> dict cấu hình chung của nhóm."""\n    with open(duong_dan, encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef chuan_hoa_nfc(cac_tu):\n    """Chuẩn hóa Unicode NFC cho danh sách token (không đổi số lượng token)."""\n    return [unicodedata.normalize("NFC", tu) for tu in cac_tu]\n\n\ndef doc_file_json(duong_dan, nfc=True):\n    """Đọc 1 file .json của PhoNER_COVID19 (mỗi dòng {"words": [...], "tags": [...]}).\n    Đầu ra: (cac_cau, cac_nhan) - 2 list cùng độ dài; mỗi phần tử là list token / list nhãn của 1 câu."""\n    cac_cau, cac_nhan = [], []\n    with open(duong_dan, encoding="utf-8") as f:\n        for dong in f:\n            if not dong.strip():\n                continue\n            mau = json.loads(dong)\n            cac_cau.append(chuan_hoa_nfc(mau["words"]) if nfc else mau["words"])\n            cac_nhan.append(mau["tags"])\n    return cac_cau, cac_nhan\n\n\ndef tai_du_lieu(muc="word", thu_muc_data=None, nfc=True):\n    """Đọc đủ 3 tập của 1 phiên bản dữ liệu.\n    muc = "word"     (đã tách từ bằng RDRSegmenter - dùng cho PhoBERT, CRF)\n        = "syllable" (âm tiết - dùng cho XLM-R, mBERT, zero-shot, PhoBERT không tách từ)\n    Đầu ra: {"train": (cac_cau, cac_nhan), "dev": (...), "test": (...)}"""\n    if muc not in ("word", "syllable"):\n        raise ValueError("muc phải là \'word\' hoặc \'syllable\'")\n    thu_muc_data = thu_muc_data or os.path.join(THU_MUC_GOC, "data")\n    du_lieu = {}\n    for tap in CAC_TAP:\n        duong_dan = os.path.join(thu_muc_data, muc, f"{tap}_{muc}.json")\n        if not os.path.exists(duong_dan):\n            raise FileNotFoundError(f"Không thấy {duong_dan}. Xem data/README.md để tải dữ liệu PhoNER_COVID19.")\n        du_lieu[tap] = doc_file_json(duong_dan, nfc=nfc)\n    return du_lieu\n\n\ndef tao_label2id(cau_hinh=None):\n    """label2id, id2label CỐ ĐỊNH theo thứ tự khóa "nhan" trong configs/common.json (21 nhãn).\n    Cả nhóm phải dùng đúng hàm này để id của nhãn giống nhau ở mọi mô hình."""\n    cau_hinh = cau_hinh or doc_cau_hinh()\n    label2id = {nhan: i for i, nhan in enumerate(cau_hinh["nhan"])}\n    id2label = {i: nhan for nhan, i in label2id.items()}\n    return label2id, id2label\n\n\ndef lay_thuc_the(nhan_cau):\n    """Lấy danh sách thực thể (loai, vi_tri_dau, vi_tri_cuoi) từ dãy nhãn BIO (IOB2, vị trí tính cả 2 đầu).\n    Viết tay (không cần seqeval) để file này chạy được ở mọi máy."""\n    thuc_the, loai_hien_tai, dau = [], None, None\n    for i, nhan in enumerate(list(nhan_cau) + ["O"]):\n        bat_dau_moi = nhan.startswith("B-") or nhan == "O" or (nhan.startswith("I-") and nhan[2:] != loai_hien_tai)\n        if bat_dau_moi and loai_hien_tai is not None:\n            thuc_the.append((loai_hien_tai, dau, i - 1))\n            loai_hien_tai = None\n        if nhan.startswith("B-") or (nhan.startswith("I-") and loai_hien_tai is None):\n            loai_hien_tai, dau = nhan[2:], i\n    return thuc_the\n\n\ndef kiem_tra_du_lieu(du_lieu, label2id):\n    """Kiểm tra từng tập: số token = số nhãn, nhãn nằm trong label2id, nhãn I-X đứng sai chỗ (vi phạm IOB2).\n    Đầu ra: {tap: {"so_cau", "lech_do_dai", "nhan_la", "I_sai_cho", "so_thuc_the"}}"""\n    ket_qua = {}\n    for tap, (cac_cau, cac_nhan) in du_lieu.items():\n        lech = sum(len(c) != len(n) for c, n in zip(cac_cau, cac_nhan))\n        nhan_la = {x for n in cac_nhan for x in n if x not in label2id}\n        i_sai = 0\n        for n in cac_nhan:\n            truoc = "O"\n            for x in n:\n                if x.startswith("I-") and truoc[2:] != x[2:]:\n                    i_sai += 1\n                truoc = x\n        ket_qua[tap] = {"so_cau": len(cac_cau), "lech_do_dai": lech, "nhan_la": sorted(nhan_la),\n                        "I_sai_cho": i_sai, "so_thuc_the": sum(len(lay_thuc_the(n)) for n in cac_nhan)}\n    return ket_qua\n\n\ndef dem_thuc_the_theo_loai(cac_nhan):\n    """Counter {loai: so_thuc_the} cho 1 tập."""\n    return Counter(loai for n in cac_nhan for loai, _, _ in lay_thuc_the(n))\n\n\nif __name__ == "__main__":\n    cau_hinh = doc_cau_hinh()\n    label2id, id2label = tao_label2id(cau_hinh)\n    print("Số nhãn:", len(label2id), "(mong đợi 21)")\n    for muc in ["word", "syllable"]:\n        du_lieu = tai_du_lieu(muc)\n        print(f"\\n[{muc}] mong đợi {cau_hinh[\'du_lieu\'][\'so_cau_mong_doi\']}")\n        for tap, kq in kiem_tra_du_lieu(du_lieu, label2id).items():\n            print(f"  {tap:5s} {kq}")\n', 'src/aggregate.py': '"""aggregate.py - Gộp các file results/*.json thành bảng mean ± std + biểu đồ cho báo cáo.\n\nNgười phụ trách: Hoàng Duyên · Issue T20 (bản đầu) → T27 + T39 (bản đầy đủ, biểu đồ)\nChạy được trên máy cá nhân (VS Code) hoặc Colab, KHÔNG cần GPU:   python src/aggregate.py\n\nĐầu ra (thư mục results/tong_hop/):\n  bang_tong_hop.csv / .md     mean ± std (micro/macro F1, P, R) của từng phương pháp + số liệu bài báo để đối chiếu\n  bang_theo_seed.csv / .md    từng seed của từng kiến trúc\n  bang_f1_tung_loai.csv / .md F1 từng loại thực thể (mean qua seed)\n  kiem_dinh.md                kiểm định Welch t-test giữa các kiến trúc (kèm cảnh báo n = 3)\n  hinh_f1_kien_truc.png       cột mean ± std\n  hinh_f1_tung_loai.png       F1 từng loại\nQuy ước: std MẪU (chia n-1, numpy ddof=1). CRF và zero-shot chỉ có 1 lượt -> không có std.\n"""\n\nimport glob\nimport json\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nTHU_MUC_GOC = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nTHU_MUC_KQ = os.path.join(THU_MUC_GOC, "results")\nTHU_MUC_RA = os.path.join(THU_MUC_KQ, "tong_hop")\nCAC_LOAI = ["PATIENT_ID", "NAME", "AGE", "GENDER", "JOB", "LOCATION", "ORGANIZATION",\n            "SYMPTOM_AND_DISEASE", "TRANSPORTATION", "DATE"]\nTEN_HIEN_THI = {"crf": "(A) CRF", "zeroshot": "(B) Zero-shot", "phobert": "(C) PhoBERT-base",\n                "xlmr": "(C) XLM-R-base", "mbert": "(C) mBERT-cased",\n                "phobert_khong_tach_tu": "Nâng cao: PhoBERT không tách từ"}\nTHU_TU = list(TEN_HIEN_THI)\n# Số liệu bài báo gốc (Truong et al., NAACL 2021, Bảng 3 — test, trung bình 5 seed) để đối chiếu\nBAI_BAO = [("Bài báo: BiLSTM-CNN-CRF (mức từ)", 0.910, 0.875), ("Bài báo: XLM-R base (âm tiết)", 0.925, 0.879),\n           ("Bài báo: PhoBERT base (mức từ)", 0.942, 0.920), ("Bài báo: PhoBERT large (mức từ)", 0.945, 0.931)]\n\n\ndef doc_ket_qua(thu_muc=THU_MUC_KQ):\n    """Đọc mọi results/*.json (bỏ file bắt đầu bằng "_"). Đầu ra: list dict có thêm khóa \'khoa\'."""\n    ds = []\n    for p in sorted(glob.glob(os.path.join(thu_muc, "*.json"))):\n        ten = os.path.basename(p)\n        if ten.startswith("_"):\n            continue\n        kq = json.load(open(p, encoding="utf-8"))\n        kq["khoa"] = kq.get("khoa_mo_hinh") or os.path.splitext(ten)[0]     # crf.json, zeroshot.json của Phúc\n        if kq.get("thu_nhanh"):\n            continue\n        ds.append(kq)\n    return ds\n\n\ndef dinh_dang(gia_tri):\n    m = np.mean(gia_tri)\n    if len(gia_tri) < 2:\n        return f"{m:.4f}"\n    return f"{m:.4f} ± {np.std(gia_tri, ddof=1):.4f}"\n\n\ndef bang_markdown(df):\n    cot = list(df.columns)\n    dong = ["| " + " | ".join(cot) + " |", "|" + "|".join(["---"] * len(cot)) + "|"]\n    dong += ["| " + " | ".join(str(x) for x in r) + " |" for r in df.itertuples(index=False)]\n    return "\\n".join(dong)\n\n\ndef tong_hop(ds):\n    nhom = {}\n    for kq in ds:\n        nhom.setdefault(kq["khoa"], []).append(kq)\n    khoa_sap = sorted(nhom, key=lambda k: THU_TU.index(k) if k in THU_TU else 99)\n\n    dong_tong, dong_seed, dong_loai = [], [], []\n    for k in khoa_sap:\n        g = sorted(nhom[k], key=lambda x: x.get("seed", 0))\n        dong_tong.append({\n            "Phương pháp": TEN_HIEN_THI.get(k, k), "Số lượt (seed)": len(g),\n            "Micro-F1": dinh_dang([x["f1_micro"] for x in g]), "Macro-F1": dinh_dang([x["f1_macro"] for x in g]),\n            "Precision": dinh_dang([x["precision_micro"] for x in g]), "Recall": dinh_dang([x["recall_micro"] for x in g]),\n            "_mean": float(np.mean([x["f1_micro"] for x in g])),\n            "_std": float(np.std([x["f1_micro"] for x in g], ddof=1)) if len(g) > 1 else 0.0,\n        })\n        for x in g:\n            if "seed" in x:\n                dong_seed.append({"Kiến trúc": TEN_HIEN_THI.get(k, k), "Seed": x["seed"], "Micro-F1": x["f1_micro"],\n                                  "Macro-F1": x["f1_macro"], "F1 dev": x.get("f1_micro_dev"),\n                                  "Epoch tốt nhất": x.get("best_epoch"), "Thời gian (phút)": round(x.get("thoi_gian_train_giay", 0) / 60, 1)})\n        dong_loai.append({"Phương pháp": TEN_HIEN_THI.get(k, k),\n                          **{l: round(float(np.mean([x["f1_tung_loai"].get(l, 0) for x in g])), 4) for l in CAC_LOAI}})\n    for ten, mi, ma in BAI_BAO:\n        dong_tong.append({"Phương pháp": ten, "Số lượt (seed)": 5, "Micro-F1": f"{mi:.3f}", "Macro-F1": f"{ma:.3f}",\n                          "Precision": "—", "Recall": "—", "_mean": mi, "_std": 0.0})\n    return nhom, pd.DataFrame(dong_tong), pd.DataFrame(dong_seed), pd.DataFrame(dong_loai)\n\n\ndef kiem_dinh(nhom):\n    """Welch t-test trên Micro-F1 giữa các cặp kiến trúc fine-tune (Dror et al., ACL 2018 khuyến nghị kiểm định\n    ý nghĩa thống kê; với n = 3 seed, độ mạnh kiểm định thấp -> chỉ dùng tham khảo)."""\n    try:\n        from scipy.stats import ttest_ind\n    except ImportError:\n        return "Chưa cài scipy nên bỏ qua kiểm định."\n    cac_kt = [k for k in ("phobert", "xlmr", "mbert", "phobert_khong_tach_tu") if len(nhom.get(k, [])) >= 2]\n    dong = ["| Cặp so sánh | Chênh lệch mean Micro-F1 | p-value (Welch) | Kết luận ở mức 0,05 |", "|---|---|---|---|"]\n    for i in range(len(cac_kt)):\n        for j in range(i + 1, len(cac_kt)):\n            a = [x["f1_micro"] for x in nhom[cac_kt[i]]]\n            b = [x["f1_micro"] for x in nhom[cac_kt[j]]]\n            p = ttest_ind(a, b, equal_var=False).pvalue\n            dong.append(f"| {TEN_HIEN_THI[cac_kt[i]]} vs {TEN_HIEN_THI[cac_kt[j]]} | {np.mean(a) - np.mean(b):+.4f} | "\n                        f"{p:.4f} | {\'khác biệt có ý nghĩa\' if p < 0.05 else \'chưa đủ bằng chứng khác biệt\'} |")\n    return "\\n".join(dong) + "\\n\\n*Lưu ý: chỉ 3 seed/kiến trúc nên kiểm định có độ mạnh thấp; kết luận cần thận trọng.*"\n\n\ndef ve_hinh(df_tong, df_loai):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    os.makedirs(THU_MUC_RA, exist_ok=True)\n    d = df_tong[~df_tong["Phương pháp"].str.startswith("Bài báo")]\n    fig, ax = plt.subplots(figsize=(8, 4))\n    ax.bar(d["Phương pháp"], d["_mean"], yerr=d["_std"], capsize=6, color="#2b6cb0")\n    for i, (m, s) in enumerate(zip(d["_mean"], d["_std"])):\n        ax.text(i, m + s + 0.005, f"{m:.3f}", ha="center", fontsize=9)\n    ax.set_ylim(max(0, d["_mean"].min() - 0.1), 1.0)\n    ax.set_ylabel("Micro-F1 (test)")\n    ax.set_title("Micro-F1 trên tập test (thanh lỗi = độ lệch chuẩn qua 3 seed)")\n    plt.xticks(rotation=20, ha="right")\n    plt.tight_layout()\n    fig.savefig(os.path.join(THU_MUC_RA, "hinh_f1_kien_truc.png"), dpi=200)\n    plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(11, 4.5))\n    w = 0.8 / max(1, len(df_loai))\n    x = np.arange(len(CAC_LOAI))\n    for i, r in enumerate(df_loai.itertuples(index=False)):\n        ax.bar(x + i * w, [getattr(r, l) for l in CAC_LOAI], w, label=r[0])\n    ax.set_xticks(x + w * (len(df_loai) - 1) / 2)\n    ax.set_xticklabels(CAC_LOAI, rotation=30, ha="right")\n    ax.set_ylabel("F1 (test)")\n    ax.set_title("F1 theo từng loại thực thể")\n    ax.legend(fontsize=8, ncol=3)\n    plt.tight_layout()\n    fig.savefig(os.path.join(THU_MUC_RA, "hinh_f1_tung_loai.png"), dpi=200)\n    plt.close(fig)\n\n\ndef main():\n    ds = doc_ket_qua()\n    if not ds:\n        print("Chưa có file kết quả nào trong results/.")\n        return\n    nhom, df_tong, df_seed, df_loai = tong_hop(ds)\n    os.makedirs(THU_MUC_RA, exist_ok=True)\n    cot = [c for c in df_tong.columns if not c.startswith("_")]\n    for ten, df in (("bang_tong_hop", df_tong[cot]), ("bang_theo_seed", df_seed), ("bang_f1_tung_loai", df_loai)):\n        df.to_csv(os.path.join(THU_MUC_RA, ten + ".csv"), index=False, encoding="utf-8-sig")\n        open(os.path.join(THU_MUC_RA, ten + ".md"), "w", encoding="utf-8").write(bang_markdown(df))\n    open(os.path.join(THU_MUC_RA, "kiem_dinh.md"), "w", encoding="utf-8").write(kiem_dinh(nhom))\n    ve_hinh(df_tong, df_loai)\n    print("== Bảng tổng hợp ==\\n" + bang_markdown(df_tong[cot]))\n    print("\\n== Theo seed ==\\n" + bang_markdown(df_seed))\n    print("\\n== Kiểm định ==\\n" + kiem_dinh(nhom))\n    print(f"\\nĐã ghi bảng và hình vào {THU_MUC_RA}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/phan_tich_loi.py': '"""phan_tich_loi.py - Phân tích lỗi có hệ thống cho 3 kiến trúc (và CRF nếu có) trên tập test.\n\nNgười phụ trách: Đức Tâm · Issue T31 + T38 (đọc lỗi từng model: T32 Duyên, T33 Tâm, T34 Trinh)\nChạy trên máy cá nhân (VS Code) hoặc Colab, KHÔNG cần GPU:   python src/phan_tich_loi.py --seed 42\nCRF và zero-shot (của Phúc) tự được đưa vào nếu có file du_doan_test.json (xem FILE_CRF_ZEROSHOT).\n\nPhân loại lỗi theo đúng khung của bài báo gốc (Truong et al., 2021, mục 5 — Error analysis):\n  1. Sai loại      : đúng ranh giới, sai loại thực thể (vd. LOCATION ↔ ORGANIZATION)\n  2. Sai biên      : đúng loại, lệch ranh giới (thiếu/thừa âm tiết)\n  3. Sai cả hai    : chồng lấn nhưng sai cả biên lẫn loại\n  4. Bỏ sót        : thực thể thật không được dự đoán (không chồng lấn với dự đoán nào)\n  5. Dự đoán thừa  : dự đoán thực thể ở chỗ nhãn thật là O\nSo sánh công bằng giữa mô hình mức TỪ (PhoBERT) và mức ÂM TIẾT (XLM-R, mBERT): mọi thực thể được quy đổi về\nvị trí ÂM TIẾT (tách "Bệnh_viện" -> "Bệnh", "viện"), nên cùng một thực thể có cùng tọa độ ở mọi mô hình.\n\nĐầu ra (results/phan_tich_loi/): bang_nhom_loi.csv/.md, ma_tran_nham_loai.csv, loi_chung_rieng.md,\nvi_du_<nhom>.csv (câu thật), hinh_nhom_loi.png, tom_tat.md\n"""\n\nimport argparse\nimport json\nimport os\nimport sys\nfrom collections import Counter, defaultdict\n\nTHU_MUC_SRC = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, THU_MUC_SRC)\nfrom data import THU_MUC_GOC, lay_thuc_the, tai_du_lieu  # noqa: E402\n\nimport pandas as pd  # noqa: E402\n\nTHU_MUC_RA = os.path.join(THU_MUC_GOC, "results", "phan_tich_loi")\nTEN = {"phobert": "PhoBERT", "xlmr": "XLM-R", "mbert": "mBERT", "crf": "CRF", "zeroshot": "Zero-shot", "phobert_khong_tach_tu": "PhoBERT-không-tách-từ"}\nNHOM_LOI = ["Sai loại", "Sai biên", "Sai cả hai", "Bỏ sót", "Dự đoán thừa"]\nTEN_FILE = {"Sai loại": "sai_loai", "Sai biên": "sai_bien", "Sai cả hai": "sai_ca_hai", "Bỏ sót": "bo_sot",\n            "Dự đoán thừa": "du_doan_thua"}\nLOAI_DAC_THU = ["PATIENT_ID", "ORGANIZATION", "JOB", "SYMPTOM_AND_DISEASE"]\n\n\ndef tu_sang_am_tiet(cac_tu):\n    """Chỉ số âm tiết bắt đầu của từng từ + tổng số âm tiết. \'Bệnh_viện\' = 2 âm tiết."""\n    dau, n = [], 0\n    for tu in cac_tu:\n        dau.append(n)\n        n += len(tu.split("_"))\n    return dau, n\n\n\ndef thuc_the_am_tiet(nhan, cac_tu=None):\n    """Tập thực thể (loai, dau, cuoi) theo vị trí ÂM TIẾT. cac_tu=None nghĩa là dữ liệu đã ở mức âm tiết."""\n    tt = lay_thuc_the(nhan)\n    if cac_tu is None:\n        return set(tt)\n    dau, _ = tu_sang_am_tiet(cac_tu)\n    return {(l, dau[s], dau[e] + len(cac_tu[e].split("_")) - 1) for l, s, e in tt}\n\n\ndef chong_lan(a, b):\n    return a[1] <= b[2] and b[1] <= a[2]\n\n\ndef phan_loai_loi(vang, doan):\n    """vang, doan: tập (loai, dau, cuoi). Trả về list (nhom_loi, thuc_the_vang | None, thuc_the_doan | None)."""\n    loi = []\n    for g in sorted(vang - doan, key=lambda x: (x[1], x[2], x[0])):\n        cac_p = [p for p in doan if chong_lan(g, p)]\n        if not cac_p:\n            loi.append(("Bỏ sót", g, None))\n            continue\n        # giống cách của Phúc (notebook baseline): lấy dự đoán chồng lấn ĐỨNG TRƯỚC NHẤT trong câu\n        p = min(cac_p, key=lambda p: (p[1], p[2], p[0]))\n        cung_bien, cung_loai = (g[1], g[2]) == (p[1], p[2]), g[0] == p[0]\n        loi.append(("Sai loại" if cung_bien else "Sai biên" if cung_loai else "Sai cả hai", g, p))\n    for p in sorted(doan - vang, key=lambda x: (x[1], x[2], x[0])):\n        if not any(chong_lan(p, g) for g in vang):\n            loi.append(("Dự đoán thừa", None, p))\n    return loi\n\n\n# File dự đoán test của Phúc (notebook Baseline_CRF_ZeroShot) có khóa crf_tach_tu (mức từ) và zero_shot (âm tiết)\nFILE_CRF_ZEROSHOT = ["results/du_doan/du_doan_test.json", "results/phan_tich/du_doan_test.json", "ket_qua/du_doan_test.json",\n                     "results/du_doan_test.json"]\nKHOA_PHUC = {"crf": ("crf_tach_tu", "word"), "zeroshot": ("zero_shot", "syllable")}\n\n\ndef doc_du_doan(khoa, seed):\n    """Đọc dự đoán test của 1 mô hình. Fine-tune: results/du_doan/{khoa}_seed{seed}.json (train_ner.py ghi).\n    CRF / zero-shot (chỉ 1 lượt, không có seed): đọc từ file du_doan_test.json của Phúc."""\n    p = os.path.join(THU_MUC_GOC, "results", "du_doan", f"{khoa}_seed{seed}.json")\n    if os.path.exists(p):\n        return json.load(open(p, encoding="utf-8"))\n    if khoa in KHOA_PHUC:\n        for f in FILE_CRF_ZEROSHOT:\n            f = os.path.join(THU_MUC_GOC, f)\n            if os.path.exists(f):\n                d = json.load(open(f, encoding="utf-8"))\n                khoa_file, muc = KHOA_PHUC[khoa]\n                if khoa_file in d:\n                    return {"khoa_mo_hinh": khoa, "muc_du_lieu": muc, "du_doan": d[khoa_file]}\n    return None\n\n\ndef to_dam(cac_am_tiet, dau, cuoi):\n    return " ".join(cac_am_tiet[:dau] + ["【" + " ".join(cac_am_tiet[dau:cuoi + 1]) + "】"] + cac_am_tiet[cuoi + 1:])\n\n\ndef main(seed=42, cac_khoa=("crf", "zeroshot", "phobert", "xlmr", "mbert"), so_vi_du=5):\n    du_lieu = {"word": tai_du_lieu("word")["test"], "syllable": tai_du_lieu("syllable")["test"]}\n    cau_am_tiet, nhan_am_tiet = du_lieu["syllable"]\n    # Nhãn thật của ĐÚNG phiên bản mô hình dùng (mức từ hoặc âm tiết), rồi quy về vị trí âm tiết.\n    # (vài thực thể ở bản mức từ lệch biên so với bản âm tiết do tách từ -> chấm theo bản của chính mô hình,\n    #  khớp với số của seqeval và của notebook Phúc)\n    vang_theo_muc = {"syllable": [thuc_the_am_tiet(n) for n in nhan_am_tiet],\n                     "word": [thuc_the_am_tiet(n, c) for c, n in zip(*du_lieu["word"])]}\n    vang = vang_theo_muc["syllable"]\n    os.makedirs(THU_MUC_RA, exist_ok=True)\n\n    dem, nham_loai, loi_cua = {}, {}, {}\n    vi_du = defaultdict(list)\n    for khoa in cac_khoa:\n        dd = doc_du_doan(khoa, seed)\n        if dd is None:\n            print(f"  (bỏ qua {khoa}: chưa có results/du_doan/{khoa}_seed{seed}.json)")\n            continue\n        muc = dd["muc_du_lieu"]\n        cau_goc = du_lieu[muc][0]\n        dem[khoa], nham_loai[khoa], loi_cua[khoa] = Counter(), Counter(), set()\n        for i, nhan in enumerate(dd["du_doan"]):\n            doan = thuc_the_am_tiet(nhan, cau_goc[i] if muc == "word" else None)\n            for nhom, g, p in phan_loai_loi(vang_theo_muc[muc][i], doan):\n                dem[khoa][nhom] += 1\n                if nhom == "Sai loại":\n                    nham_loai[khoa][(g[0], p[0])] += 1\n                if g is not None:\n                    loi_cua[khoa].add((i, g))                        # thực thể thật mà model này làm sai\n                if len(vi_du[(khoa, nhom)]) < 200:\n                    tt = g or p\n                    vi_du[(khoa, nhom)].append({"cau_so": i, "mo_hinh": TEN.get(khoa, khoa), "nhom_loi": nhom,\n                                                 "that": f"{g[0]}: {\' \'.join(cau_am_tiet[i][g[1]:g[2] + 1])}" if g else "O",\n                                                 "du_doan": f"{p[0]}: {\' \'.join(cau_am_tiet[i][p[1]:p[2] + 1])}" if p else "O",\n                                                 "cau": to_dam(cau_am_tiet[i], tt[1], tt[2])})\n    if not dem:\n        print("Chưa có file dự đoán nào. Chạy notebook huấn luyện trước.")\n        return\n\n    # 1) Bảng số lỗi theo nhóm × mô hình\n    tong_vang = sum(len(v) for v in vang)\n    bang = pd.DataFrame([{"Nhóm lỗi": n, **{TEN[k]: dem[k][n] for k in dem}} for n in NHOM_LOI])\n    bang.loc[len(bang)] = ["Tổng lỗi", *[sum(dem[k].values()) for k in dem]]\n    bang.to_csv(os.path.join(THU_MUC_RA, "bang_nhom_loi.csv"), index=False, encoding="utf-8-sig")\n\n    # 2) Ma trận nhầm loại (chỉ nhóm "Sai loại")\n    dong = [{"Mô hình": TEN[k], "Thật → Dự đoán": f"{a} → {b}", "Số lần": c}\n            for k in nham_loai for (a, b), c in nham_loai[k].most_common(8)]\n    pd.DataFrame(dong).to_csv(os.path.join(THU_MUC_RA, "ma_tran_nham_loai.csv"), index=False, encoding="utf-8-sig")\n\n    # 3) Lỗi chung / lỗi riêng: 3 kiến trúc có mắc CÙNG lỗi không?\n    cac_k = [k for k in loi_cua if k in ("phobert", "xlmr", "mbert")] or list(loi_cua)   # so sánh 3 kiến trúc fine-tune\n    chung_tat_ca = set.intersection(*[loi_cua[k] for k in cac_k]) if len(cac_k) > 1 else set()\n    dong_cr = [f"- Thực thể thật bị **mọi** mô hình ({\', \'.join(TEN[k] for k in cac_k)}) làm sai: **{len(chung_tat_ca)}**"]\n    for k in cac_k:\n        rieng = loi_cua[k] - set().union(*[loi_cua[j] for j in cac_k if j != k])  # noqa: E501\n        dong_cr.append(f"- Chỉ riêng {TEN[k]} sai: {len(rieng)} (tổng {TEN[k]} sai {len(loi_cua[k])})")\n    loai_chung = Counter(g[0] for _, g in chung_tat_ca)\n    dong_cr.append(f"- Loại thực thể hay bị cả {len(cac_k)} mô hình cùng sai: " + ", ".join(f"{l} ({c})" for l, c in loai_chung.most_common(5)))\n    open(os.path.join(THU_MUC_RA, "loi_chung_rieng.md"), "w", encoding="utf-8").write("\\n".join(dong_cr))\n\n    # 4) Ví dụ thật cho từng nhóm lỗi (mỗi nhóm mỗi mô hình 1 file gộp)\n    for nhom in NHOM_LOI:\n        dong_vd = [x for k in dem for x in vi_du[(k, nhom)][:so_vi_du * 4]]\n        pd.DataFrame(dong_vd).to_csv(os.path.join(THU_MUC_RA, f"vi_du_{TEN_FILE[nhom]}.csv"),\n                                     index=False, encoding="utf-8-sig")\n\n    # 5) Thực thể đặc thù: tỉ lệ sai theo loại\n    sai_theo_loai = pd.DataFrame([{"Loại": l, "Số thực thể thật": sum(1 for v in vang for g in v if g[0] == l),\n                                   **{TEN[k]: sum(1 for _, g in loi_cua[k] if g[0] == l) for k in loi_cua}}\n                                  for l in LOAI_DAC_THU])\n    sai_theo_loai.to_csv(os.path.join(THU_MUC_RA, "loi_thuc_the_dac_thu.csv"), index=False, encoding="utf-8-sig")\n\n    # 6) Hình\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    ax = bang[bang["Nhóm lỗi"] != "Tổng lỗi"].set_index("Nhóm lỗi").plot(kind="bar", figsize=(9, 4), rot=0)\n    ax.set_ylabel("Số lỗi trên test")\n    ax.set_title(f"Số lỗi theo nhóm và mô hình (seed {seed})")\n    plt.tight_layout()\n    plt.savefig(os.path.join(THU_MUC_RA, "hinh_nhom_loi.png"), dpi=200)\n    plt.close()\n\n    def md(df):\n        cot = list(df.columns)\n        return "\\n".join(["| " + " | ".join(map(str, cot)) + " |", "|" + "---|" * len(cot)] +\n                         ["| " + " | ".join(map(str, r)) + " |" for r in df.itertuples(index=False)])\n    tom_tat = [f"# Tóm tắt phân tích lỗi (test, seed {seed}; {tong_vang} thực thể thật)", "", "## Số lỗi theo nhóm", md(bang),\n               "", "## Nhầm loại hay gặp nhất", md(pd.DataFrame(dong)) if dong else "(không có)",\n               "", "## Lỗi chung và lỗi riêng", *dong_cr, "", "## Thực thể đặc thù", md(sai_theo_loai)]\n    open(os.path.join(THU_MUC_RA, "tom_tat.md"), "w", encoding="utf-8").write("\\n".join(tom_tat))\n    open(os.path.join(THU_MUC_RA, "bang_nhom_loi.md"), "w", encoding="utf-8").write(md(bang))\n    print("\\n".join(tom_tat))\n    print(f"\\nĐã ghi kết quả vào {THU_MUC_RA}")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--mo_hinh", nargs="+", default=["crf", "zeroshot", "phobert", "xlmr", "mbert"])\n    a = ap.parse_args()\n    main(a.seed, a.mo_hinh)\n', 'src/xuat_du_doan.py': '"""xuat_du_doan.py - Gộp dự đoán test của mọi mô hình vào 1 file CSV dễ đọc (Issue T27, đầu vào cho T31–T35).\n\nNgười phụ trách: Hoàng Duyên · Chạy trên máy (VS Code), không cần GPU:   python src/xuat_du_doan.py --seed 42\nĐầu ra: results/du_doan/predictions_test.csv — 3.000 dòng (1 dòng = 1 câu test), các cột:\n  cau_so | cau (âm tiết) | that | crf | zeroshot | phobert | xlmr | mbert | so_mo_hinh_sai\nMỗi ô là danh sách thực thể "LOẠI: văn bản" nối bằng " ; ". Mô hình mức từ (PhoBERT, CRF) được quy về âm tiết\n("Bệnh_viện" = 2 âm tiết) để mọi cột cùng đơn vị.\n"""\n\nimport argparse\nimport os\nimport sys\n\nTHU_MUC_SRC = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, THU_MUC_SRC)\nfrom data import THU_MUC_GOC, tai_du_lieu  # noqa: E402\nfrom phan_tich_loi import TEN, doc_du_doan, thuc_the_am_tiet  # noqa: E402\n\nimport pandas as pd  # noqa: E402\n\n\ndef dinh_dang(tap_tt, cac_am_tiet):\n    return " ; ".join(f"{l}: {\' \'.join(cac_am_tiet[s:e + 1])}" for l, s, e in sorted(tap_tt, key=lambda x: (x[1], x[2])))\n\n\ndef main(seed=42, cac_khoa=("crf", "zeroshot", "phobert", "xlmr", "mbert")):\n    tu, am = tai_du_lieu("word")["test"], tai_du_lieu("syllable")["test"]\n    cau_am, nhan_am = am\n    vang = [thuc_the_am_tiet(n) for n in nhan_am]\n    cot = {}\n    for khoa in cac_khoa:\n        dd = doc_du_doan(khoa, seed)\n        if dd is None:\n            print(f"  (bỏ qua {khoa}: chưa có file dự đoán)")\n            continue\n        if len(dd["du_doan"]) != len(cau_am):\n            raise ValueError(f"{khoa}: có {len(dd[\'du_doan\'])} câu, mong đợi {len(cau_am)}")\n        cot[khoa] = [thuc_the_am_tiet(n, tu[0][i] if dd["muc_du_lieu"] == "word" else None)\n                     for i, n in enumerate(dd["du_doan"])]\n    dong = []\n    for i in range(len(cau_am)):\n        r = {"cau_so": i, "cau": " ".join(cau_am[i]), "that": dinh_dang(vang[i], cau_am[i])}\n        for k, v in cot.items():\n            r[k] = dinh_dang(v[i], cau_am[i])\n        r["so_mo_hinh_sai"] = sum(v[i] != vang[i] for v in cot.values())\n        dong.append(r)\n    df = pd.DataFrame(dong)\n    ra = os.path.join(THU_MUC_GOC, "results", "du_doan", "predictions_test.csv")\n    os.makedirs(os.path.dirname(ra), exist_ok=True)\n    df.to_csv(ra, index=False, encoding="utf-8-sig")\n    print(f"Đã ghi {len(df)} câu × {len(cot)} mô hình ({\', \'.join(TEN.get(k, k) for k in cot)}) -> {ra}")\n    print("Số câu mọi mô hình đều đúng:", int((df["so_mo_hinh_sai"] == 0).sum()))\n    return df\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--seed", type=int, default=42)\n    main(ap.parse_args().seed)\n'}

import os, sys, glob, zipfile, shutil, subprocess
TREN_COLAB = "google.colab" in sys.modules
if TREN_COLAB:
    os.makedirs("/content/de26", exist_ok=True); os.chdir("/content/de26")
    for f, nd in MA_NGUON.items():                       # Colab không có repo -> ghi code của nhóm
        os.makedirs(os.path.dirname(f), exist_ok=True); open(f, "w", encoding="utf-8").write(nd)
    if not os.path.exists("data/word/train_word.json"):  # dữ liệu công khai của VinAI
        if not os.path.isdir("/tmp/PhoNER_COVID19/data/word"):
            subprocess.run("git clone -q --depth 1 https://github.com/VinAIResearch/PhoNER_COVID19.git /tmp/PhoNER_COVID19", shell=True)
        subprocess.run("mkdir -p data && cp -r /tmp/PhoNER_COVID19/data/word /tmp/PhoNER_COVID19/data/syllable data/", shell=True)
    for z in sorted(set(glob.glob("/content/ket_qua_*.zip") + glob.glob("/content/de26/ket_qua_*.zip"))):
        zipfile.ZipFile(z).extractall("."); print("✅ giải nén", os.path.basename(z))
    os.makedirs("results/du_doan", exist_ok=True)
    for f in glob.glob("/content/*.json"):               # file của Phúc kéo thẳng vào Files
        ten = os.path.basename(f)
        dich = "results/du_doan/" + ten if ten == "du_doan_test.json" else "results/" + ten
        shutil.copy(f, dich); print("✅ chép", ten, "->", dich)
else:                                                    # VS Code: tìm thư mục gốc repo (có src/aggregate.py)
    d = os.getcwd()
    while d != os.path.dirname(d) and not os.path.exists(os.path.join(d, "src", "aggregate.py")):
        d = os.path.dirname(d)
    if os.path.exists(os.path.join(d, "src", "aggregate.py")):
        os.chdir(d)
    else:
        raise SystemExit("❌ Không tìm thấy thư mục repo. Hãy mở notebook nằm TRONG NLP-phoner-covid19-26/notebooks/ "
                         "(không mở bản trong thư mục gói GOI_17-10).")
print("Thư mục làm việc:", os.getcwd())

## 1. Kiểm tra đầu vào — còn thiếu file gì, của ai?

In [ ]:
CAN_CO = [("Phúc", "results/crf.json"), ("Phúc", "results/zeroshot.json"), ("Phúc", "results/du_doan/du_doan_test.json")]
for ai, khoa in [("Hoàng Duyên", "phobert"), ("Đức Tâm", "xlmr"), ("Yến Trinh", "mbert")]:
    CAN_CO += [(ai, f"results/{khoa}_seed{s}.json") for s in (42, 123, 2024)] + [(ai, f"results/du_doan/{khoa}_seed42.json")]
CAN_CO += [("(tải dữ liệu VinAI)", "data/word/test_word.json"), ("(tải dữ liệu VinAI)", "data/syllable/test_syllable.json")]
thieu = {}
for ai, f in CAN_CO:
    co = os.path.exists(f)
    print(("✅ " if co else "❌ ") + f)
    if not co:
        thieu.setdefault(ai, []).append(f)
CO_DU_LIEU = os.path.exists("data/word/test_word.json") and os.path.exists("data/syllable/test_syllable.json")
CO_KET_QUA = any(not os.path.basename(f).startswith("_") for f in glob.glob("results/*.json"))
CO_DU_DOAN = bool(glob.glob("results/du_doan/*_seed42.json") + glob.glob("results/du_doan/du_doan_test.json"))
if thieu:
    print("\nCÒN THIẾU — đây là nguyên nhân các ô sau báo bỏ qua:")
    for ai, ds in thieu.items():
        print(f"  • {ai}: {len(ds)} file  ->", "chép dữ liệu vào data/ (xem data/README.md)" if ai.startswith("(") else
              "nhắc bạn chạy notebook của mình rồi đưa kết quả lên GitHub (Duyên bấm Pull origin)")
else:
    print("\n✅ Đủ đầu vào. Chạy tiếp các ô dưới.")

## 2. Bảng mean ± std + kiểm định + biểu đồ (báo cáo mục 5.3)

In [ ]:
if CO_KET_QUA:
    %run src/aggregate.py
else:
    print("⏭ Bỏ qua: chưa có file nào trong results/*.json (xem ô 1).")

In [ ]:
from IPython.display import Image, display
for h in ["hinh_f1_kien_truc.png", "hinh_f1_tung_loai.png"]:
    p = f"results/tong_hop/{h}"
    display(Image(p, width=900)) if os.path.exists(p) else print("⏭ chưa có", p)

## 3. Gộp dự đoán test của mọi mô hình vào 1 CSV (T27) — đầu vào cho đọc lỗi T32–T34

In [ ]:
if CO_DU_LIEU and CO_DU_DOAN:
    %run src/xuat_du_doan.py --seed 42
else:
    print("⏭ Bỏ qua: cần dữ liệu data/ và ít nhất 1 file dự đoán trong results/du_doan/ (xem ô 1).")

## 4. Phân tích lỗi 5 nhóm × các mô hình (báo cáo mục 5.4)

In [ ]:
if CO_DU_LIEU and CO_DU_DOAN:
    %run src/phan_tich_loi.py --seed 42
else:
    print("⏭ Bỏ qua: cần dữ liệu data/ và file dự đoán (xem ô 1).")

In [ ]:
import pandas as pd
if os.path.exists("results/phan_tich_loi/hinh_nhom_loi.png"):
    display(Image("results/phan_tich_loi/hinh_nhom_loi.png", width=900))
    for nhom in ["sai_loai", "sai_bien", "bo_sot", "du_doan_thua", "sai_ca_hai"]:
        print("=====", nhom); display(pd.read_csv(f"results/phan_tich_loi/vi_du_{nhom}.csv").head(8))
else:
    print("⏭ chưa có kết quả phân tích lỗi")

## 5. Đưa kết quả lên GitHub
VS Code: GitHub Desktop sẽ thấy thư mục `results/tong_hop/`, `results/phan_tich_loi/`, `results/du_doan/predictions_test.csv` → commit trên nhánh `duyen/T20-aggregate` (hoặc `tam/T31-phan-tich-loi`) → PR. Colab: tải các thư mục đó về (chuột phải → Download) rồi chép vào repo.